## 试验与测试

In [ ]:
from mile.mile import MILE
import numpy as np
from sklearn.neural_network import MLPClassifier
from config import BalanceScale, MILEConfig
from metrics import calculate_gmean_mauc
import warnings

warnings.filterwarnings("ignore")  # 忽略警告

mile_parameter = MILEConfig(ngen=2, popsize=3, cxpb=1.0, mutpb=0.2)  # 使用默认的MILE参数
IMBALANCED_DATASET_PATH = '../datasets/mat/'  # 数据集路径
test_dataset = BalanceScale  # 数据集参数配置
file_path = IMBALANCED_DATASET_PATH + test_dataset.DATASET_NAME
name = test_dataset.DATASET_NAME.split('.')[0]  # 获取数据集的名称
mlp = MLPClassifier(hidden_layer_sizes=(test_dataset.HIDDEN_SIZE,), max_iter=test_dataset.MAX_ITER,
                    random_state=42, learning_rate_init=test_dataset.LEARNING_RATE)

mile = MILE(file_path=file_path, estimator=mlp, random_state=42, n_splits=5, display_distribution=True,
            parameter=mile_parameter)

num_run = 30
ensembles_results = []
for i in range(0, num_run):
    mile.fit()  # 进化实例子集搜索
    y_pred = mile.predict(mile.x_test)  # 预测
    y_pred_prob = mile.predict_proba(mile.x_test)  # 软标签
    gmean, mauc = calculate_gmean_mauc(y_pred_prob, mile.y_test)
    ensembles_results.append([gmean, mauc])
    print(f"第{i + 1}次运行：Gmean：{gmean}，mAUC：{mauc}")
ensembles_result_mean = np.mean(ensembles_results, axis=0)
ensembles_result_std = np.std(ensembles_results, axis=0)
print(f'集成分类结果（平均值）：{ensembles_result_mean}')
print(f'集成分类结果（标准差）：{ensembles_result_std}')

第2次运行：Gmean：0.984525，mAUC：0.994822
